# Control Effectiveness Experiment

This notebook performs the measurable baseline -> post-control risk reduction
experiment described in the project brief. It reads directly from the
synthetic CSVs generated by `data/generate_data.py`, applies the *same*
transparent risk formula used by the backend (`backend/app/risk_engine/`),
and reports baseline, target, measured result, attribution/confidence, and
error analysis.

Run `python data/generate_data.py` first if `data/processed/*.csv` does not
exist yet.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

DATA_DIR = os.path.join("..", "data", "processed")

assets = pd.read_csv(os.path.join(DATA_DIR, "assets.csv"))
controls = pd.read_csv(os.path.join(DATA_DIR, "controls.csv"))
telemetry = pd.read_csv(os.path.join(DATA_DIR, "control_telemetry.csv"))
vulnerabilities = pd.read_csv(os.path.join(DATA_DIR, "vulnerabilities.csv"))
incidents = pd.read_csv(os.path.join(DATA_DIR, "incidents.csv"))
remediation = pd.read_csv(os.path.join(DATA_DIR, "remediation.csv"))

print(f"assets={len(assets)} controls={len(controls)} telemetry={len(telemetry)} "
      f"vulnerabilities={len(vulnerabilities)} incidents={len(incidents)} remediation={len(remediation)}")


## 1. Data validation

Basic sanity checks before running the experiment -- mirrors what the
backend ingestion pipeline enforces (see `backend/app/services/ingestion.py`).


In [ ]:
issues = []

# Ensure numeric columns are actually numeric -- injected missing values
# (empty strings) make pandas infer these as object/string dtype otherwise.
telemetry["coverage_percentage"] = pd.to_numeric(telemetry["coverage_percentage"], errors="coerce")
telemetry["compliance_percentage"] = pd.to_numeric(telemetry["compliance_percentage"], errors="coerce")
vulnerabilities["cvss_score"] = pd.to_numeric(vulnerabilities["cvss_score"], errors="coerce")

missing_coverage = telemetry["coverage_percentage"].isna().sum()
missing_compliance = telemetry["compliance_percentage"].isna().sum()
issues.append(("missing coverage_percentage (post-parse)", int(missing_coverage)))
issues.append(("missing compliance_percentage (post-parse)", int(missing_compliance)))

# percentages must be within [0, 100] (checked on the numeric-parsed values)
bad_pct = telemetry[(telemetry["coverage_percentage"] < 0) | (telemetry["coverage_percentage"] > 100)]
issues.append(("invalid coverage_percentage (out of 0-100 range)", len(bad_pct)))

bad_compliance = telemetry[(telemetry["compliance_percentage"] < 0) | (telemetry["compliance_percentage"] > 100)]
issues.append(("invalid compliance_percentage (out of 0-100 range)", len(bad_compliance)))

# duplicate incident ids
dupe_incidents = incidents[incidents.duplicated("incident_id", keep=False)]
issues.append(("duplicate incident_id", dupe_incidents["incident_id"].nunique()))

# missing asset references
orphan_vulns = vulnerabilities[~vulnerabilities["asset_id"].isin(assets["asset_id"])]
issues.append(("vulnerabilities with missing asset_id reference", len(orphan_vulns)))

for label, count in issues:
    print(f"{label}: {count}")

# Clip invalid percentages and fill missing values with the column median so
# the experiment doesn't crash on bad/missing data -- same defensive
# philosophy as the backend ingestion pipeline (flag, then degrade gracefully).
for col in ["coverage_percentage", "compliance_percentage"]:
    median_val = telemetry[col].median()
    telemetry[col] = telemetry[col].fillna(median_val).clip(0, 100)


## 2. Risk scoring function (mirrors `backend/app/risk_engine/engine.py`)

Reimplemented here in pandas/numpy terms so the notebook is self-contained
and does not require importing the backend package. The formula and weights
are identical to the documented methodology in `docs/risk_methodology.md`.


In [ ]:
SEVERITY_WEIGHT = {"LOW": 0.15, "MEDIUM": 0.4, "HIGH": 0.7, "CRITICAL": 1.0}
CRITICALITY_WEIGHT = {"LOW": 0.25, "MEDIUM": 0.5, "HIGH": 0.75, "CRITICAL": 1.0}


def score_vulnerabilities(vulns_df, max_points=35.0):
    if len(vulns_df) == 0:
        return 0.0
    sev_w = vulns_df["severity"].map(SEVERITY_WEIGHT).fillna(0.15)
    is_open = vulns_df["remediation_status"].isin(["OPEN", "IN_PROGRESS", "ACCEPTED_RISK"])
    base = sev_w * np.where(is_open, 10, 2)
    base = base * np.where(vulns_df["exploit_available"] == True, 1.4, 1.0)
    base = base * np.where(vulns_df["internet_exposed"] == True, 1.25, 1.0)
    raw = base.sum()
    return min(max_points, (raw ** 0.5) * 2.1)


def score_incidents(inc_df, max_points=25.0):
    if len(inc_df) == 0:
        return 0.0
    sev_w = inc_df["severity"].map(SEVERITY_WEIGHT).fillna(0.15)
    is_open = inc_df["status"] != "RESOLVED"
    base = sev_w * np.where(is_open, 8, 3)
    raw = base.sum()
    return min(max_points, (raw ** 0.5) * 1.9)


def score_control_gap(tel_df, target_lookup, max_points=25.0):
    if len(tel_df) == 0:
        return max_points * 0.5
    target = tel_df["control_id"].map(target_lookup).fillna(95.0)
    coverage_gap = (target - tel_df["coverage_percentage"]).clip(lower=0)
    compliance_gap = (100 - tel_df["compliance_percentage"]).clip(lower=0)
    gap = (coverage_gap * 0.6 + compliance_gap * 0.4) / 100.0
    gap = gap + np.where(tel_df["health_status"] == "FAILED", 0.3,
                  np.where(tel_df["health_status"] == "DEGRADED", 0.15, 0.0))
    gap = gap.clip(upper=1.0)
    return gap.mean() * max_points


def score_asset_criticality(assets_df, max_points=15.0):
    if len(assets_df) == 0:
        return 0.0
    w = assets_df["criticality"].map(CRITICALITY_WEIGHT).fillna(0.25)
    return w.mean() * max_points


def calculate_risk(assets_df, vulns_df, inc_df, tel_df, target_lookup):
    v = score_vulnerabilities(vulns_df)
    i = score_incidents(inc_df)
    g = score_control_gap(tel_df, target_lookup)
    c = score_asset_criticality(assets_df)
    total = min(100.0, v + i + g + c)
    return {"total": round(total, 2), "vulnerability": round(v, 2), "incident": round(i, 2),
            "control_gap": round(g, 2), "asset_criticality": round(c, 2)}

target_lookup = dict(zip(controls["control_id"], controls["target_coverage"]))
print("Risk function ready.")


## 3. Define baseline and post-control periods

The synthetic generator tags telemetry and incidents with a `period` column
(`BASELINE` ~90 days ago, `CURRENT` recent). Vulnerabilities do not carry a
period column directly; we treat all discovered vulnerabilities as "present
at baseline" and only those still open/in-progress/accepted-risk as
"present now" -- i.e. remediation between baseline and now reduces the
current-period vulnerability count. This mirrors the backend's documented
assumption in `control_effectiveness.py`.


In [ ]:
tel_baseline = telemetry[telemetry["period"] == "BASELINE"]
tel_current = telemetry[telemetry["period"] == "CURRENT"]

inc_baseline = incidents[incidents["period"] == "BASELINE"]
inc_current = incidents[incidents["period"] == "CURRENT"]

vulns_baseline = vulnerabilities  # all discovered vulns considered present at baseline
vulns_current = vulnerabilities[vulnerabilities["remediation_status"].isin(["OPEN", "IN_PROGRESS", "ACCEPTED_RISK"])]

print(f"Baseline telemetry rows: {len(tel_baseline)} | Current telemetry rows: {len(tel_current)}")
print(f"Baseline incidents: {len(inc_baseline)} | Current incidents: {len(inc_current)}")
print(f"Baseline vulnerabilities (all discovered): {len(vulns_baseline)} | Current open vulnerabilities: {len(vulns_current)}")


## 4. Calculate baseline and post-control (current) organisation-wide risk

In [ ]:
risk_baseline = calculate_risk(assets, vulns_baseline, inc_baseline, tel_baseline, target_lookup)
risk_current = calculate_risk(assets, vulns_current, inc_current, tel_current, target_lookup)

print("BASELINE risk:", risk_baseline)
print("CURRENT (post-control) risk:", risk_current)


## 5. Risk reduction: absolute, percentage, target vs. measured

The project brief's worked example set a target of 20% risk reduction.
We adopt the same target here as the organisational goal and compare it
against what the data actually shows.


In [ ]:
TARGET_REDUCTION_PCT = 20.0

absolute_reduction = round(risk_baseline["total"] - risk_current["total"], 2)
pct_reduction = round((absolute_reduction / risk_baseline["total"]) * 100, 1) if risk_baseline["total"] > 0 else 0.0

print(f"Baseline risk:        {risk_baseline['total']}")
print(f"Post-control risk:    {risk_current['total']}")
print(f"Absolute reduction:   {absolute_reduction}")
print(f"Percentage reduction: {pct_reduction}%")
print(f"Target reduction:     {TARGET_REDUCTION_PCT}%")
print(f"Target achieved:      {'YES' if pct_reduction >= TARGET_REDUCTION_PCT else 'NO'}")
print(f"Difference from target: {round(pct_reduction - TARGET_REDUCTION_PCT, 1)} percentage points")


## 6. Supporting analysis: vulnerabilities, incidents, remediation, control telemetry

In [ ]:
vuln_severity_before = vulns_baseline["severity"].value_counts()
vuln_severity_after = vulns_current["severity"].value_counts()
print("Vulnerability severity -- baseline (all discovered):")
print(vuln_severity_before)
print("\nVulnerability severity -- current (still open):")
print(vuln_severity_after)


In [ ]:
inc_severity_before = inc_baseline["severity"].value_counts()
inc_severity_after = inc_current["severity"].value_counts()
print("Incident severity -- baseline period:")
print(inc_severity_before)
print("\nIncident severity -- current period:")
print(inc_severity_after)


In [ ]:
remediation_completed = remediation[remediation["status"] == "COMPLETE"]
remediation_pending = remediation[remediation["status"] != "COMPLETE"]
print(f"Remediation completed: {len(remediation_completed)}")
print(f"Remediation pending:   {len(remediation_pending)}")


In [ ]:
avg_coverage_before = tel_baseline["coverage_percentage"].mean()
avg_coverage_after = tel_current["coverage_percentage"].mean()
avg_compliance_before = tel_baseline["compliance_percentage"].mean()
avg_compliance_after = tel_current["compliance_percentage"].mean()

print(f"Avg control coverage:   {avg_coverage_before:.1f}% -> {avg_coverage_after:.1f}%")
print(f"Avg control compliance: {avg_compliance_before:.1f}% -> {avg_compliance_after:.1f}%")


## 7. Attribution and confidence

Correlation does not automatically prove causation. We compute a simple,
documented attribution/confidence score based on the *consistency* of
supporting evidence: did coverage improve, did compliance improve, did
vulnerabilities decrease, did incidents decrease? Each "yes" adds evidence
weight; the more consistent the story, the higher our confidence that the
observed risk reduction is attributable to the control improvements rather
than noise or an artifact of the scoring formula.


In [ ]:
evidence_checks = {
    "coverage_improved": avg_coverage_after > avg_coverage_before,
    "compliance_improved": avg_compliance_after > avg_compliance_before,
    "vulnerabilities_decreased": len(vulns_current) < len(vulns_baseline),
    "incidents_decreased": len(inc_current) < len(inc_baseline),
    "risk_decreased": risk_current["total"] < risk_baseline["total"],
}

for k, v in evidence_checks.items():
    print(f"{k}: {'YES' if v else 'NO'}")

attribution_confidence = round(100 * sum(evidence_checks.values()) / len(evidence_checks), 1)
print(f"\nAttribution/confidence score: {attribution_confidence}%")
print("NOTE: this reflects consistency of evidence, not statistical causal proof.")
print("A rigorous causal claim would require a controlled comparison group, which")
print("is not available for a single organisation's SOC telemetry.")


## 8. Error analysis

In [ ]:
error_notes = []

if len(tel_current) == 0:
    error_notes.append("No CURRENT telemetry found -- risk_current control-gap component defaulted to 50%.")

null_cvss = vulnerabilities["cvss_score"].isna().sum()
if null_cvss:
    error_notes.append(f"{null_cvss} vulnerabilities have missing/invalid CVSS scores (excluded from severity weighting nuance).")

out_of_range = telemetry[(telemetry["coverage_percentage"].isna())]
if len(out_of_range):
    error_notes.append(f"{len(out_of_range)} telemetry rows had missing coverage_percentage after validation.")

if not error_notes:
    error_notes.append("No significant data quality issues detected in the fields used for this experiment.")

for note in error_notes:
    print("-", note)

print("\nLimitation: the additive risk formula does not model multiplicative")
print("interaction effects (e.g., a CRITICAL vulnerability on a CRITICAL asset).")
print("See docs/risk_methodology.md for full disclosure of modeling limitations.")


## 9. Visualizations

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(["Baseline", "Post-control"], [risk_baseline["total"], risk_current["total"]],
       color=["#dc2626", "#16a34a"])
ax.set_ylabel("Risk score (0-100)")
ax.set_title("Organisation-wide risk: before vs after")
ax.set_ylim(0, 100)
plt.tight_layout()
plt.savefig("risk_before_after.png", dpi=100)
plt.show()


In [ ]:
components = ["vulnerability", "incident", "control_gap", "asset_criticality"]
before_vals = [risk_baseline[c] for c in components]
after_vals = [risk_current[c] for c in components]

x = np.arange(len(components))
width = 0.35

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - width/2, before_vals, width, label="Baseline", color="#dc2626")
ax.bar(x + width/2, after_vals, width, label="Post-control", color="#16a34a")
ax.set_xticks(x)
ax.set_xticklabels(components)
ax.set_ylabel("Points")
ax.set_title("Risk component breakdown: before vs after")
ax.legend()
plt.tight_layout()
plt.savefig("risk_components_before_after.png", dpi=100)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(["Baseline", "Current"], [avg_coverage_before, avg_coverage_after], color="#3b82f6")
ax.set_ylabel("Average control coverage (%)")
ax.set_title("Control coverage improvement")
ax.set_ylim(0, 100)
plt.tight_layout()
plt.savefig("control_coverage.png", dpi=100)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
counts = [len(vulns_baseline), len(vulns_current)]
ax.bar(["Baseline (all discovered)", "Current (still open)"], counts, color="#f97316")
ax.set_ylabel("Vulnerability count")
ax.set_title("Vulnerability reduction")
plt.tight_layout()
plt.savefig("vulnerability_reduction.png", dpi=100)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
counts = [len(inc_baseline), len(inc_current)]
ax.bar(["Baseline period", "Current period"], counts, color="#a855f7")
ax.set_ylabel("Incident count")
ax.set_title("Incident reduction")
plt.tight_layout()
plt.savefig("incident_reduction.png", dpi=100)
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
ax.bar(["Target", "Measured"], [TARGET_REDUCTION_PCT, pct_reduction],
       color=["#64748b", "#16a34a" if pct_reduction >= TARGET_REDUCTION_PCT else "#dc2626"])
ax.set_ylabel("Risk reduction (%)")
ax.set_title("Target vs. measured risk reduction")
plt.tight_layout()
plt.savefig("target_vs_measured.png", dpi=100)
plt.show()


## 10. Final conclusion

Summarize, in plain language, what the experiment found:


In [ ]:
print("=" * 70)
print("CONTROL EFFECTIVENESS EXPERIMENT -- FINAL CONCLUSION")
print("=" * 70)
print(f"Baseline risk:            {risk_baseline['total']} / 100")
print(f"Post-control risk:        {risk_current['total']} / 100")
print(f"Absolute risk reduction:  {absolute_reduction} points")
print(f"Percentage risk reduction:{pct_reduction}%")
print(f"Target:                   {TARGET_REDUCTION_PCT}% reduction")
print(f"Target achieved:          {'YES' if pct_reduction >= TARGET_REDUCTION_PCT else 'NO'}")
print(f"Attribution/confidence:   {attribution_confidence}% (evidence consistency, not proof of causation)")
print("=" * 70)
print()
print("Interpretation: control coverage and compliance improved between the")
print("baseline and current periods, accompanied by a reduction in open")
print("vulnerabilities and incidents, and a corresponding reduction in the")
print("computed business risk score. The consistency of these signals supports")
print("-- but does not statistically prove -- that the control improvements")
print("contributed to the observed risk reduction.")
